# CT4101 Machine Learning 

## Machine Learning Portfolio: Predicting 30-Day Hospital Readmission in Patients with Diabetes Using Machine Learning

## 4. Preprocessing

### Student Name: Faith Greene

### Student ID: 26256593


This notebook performs the data cleaning & preparations, based on the findings of the audit.

#### Import libraries and load the data

In [15]:
import pandas as pd
import numpy as np

raw_data = pd.read_csv(r"C:\Users\faith\OneDrive\Documents\MDX1_MSc_Health_Data_Science\CT4101_Machine_Learning\Machine_Learning_Portfolio\CT4101-Machine-Learning-Portfolio\data\raw\diabetic_data.csv")

#### Categorical Data Types

The variables encounter_id and patient_nbr should be converted to categorical variables. This will be done during the modelling pipeline.

In [16]:
categorical_codes = [
    "encounter_id",
    "patient_nbr"
]

raw_data[categorical_codes] = raw_data[categorical_codes].astype("string")

#### Missing Values

Variables with extensive missingness should be removed, based on the data audit.

In [17]:
raw_data = raw_data.drop(columns=["weight", "max_glu_serum", "A1Cresult", "medical_specialty", "payer_code"])

Other variables relating to different diagnoses should not be removed or imputed as there may be reason for such values to be missing. Further diagnoses may not always be reached.

#### Target Variable

The original categorical target variable with three categories should be converted to a bonary variable, as previously mentioned in the data audit.

In [18]:
# Create binary target
raw_data["readmitted_30"] = (
    raw_data["readmitted"] == "<30"
).astype(int)

# Remove original target
raw_data = raw_data.drop(columns=["readmitted"])

#### Features, Target Variable and Patients

Before separating into sets, the target variable and features should be identified.

In [19]:
# Target variable
y = raw_data["readmitted_30"]

# Patients
patients = raw_data["patient_nbr"]

# Features
X = raw_data.drop(
    columns=["readmitted_30", "patient_nbr"]
)


#### Training, Test & Validation Sets

Now, the data can be split into training, test and validation sets. A group aware split is carried out, to prevent data leakage. The split is 70% training, 15% validation and 15% test. A k-fold cross validation should be used in this case as a group aware split has been carried out.

In [20]:
from sklearn.model_selection import GroupShuffleSplit

# group-based split
splitter1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=123
)

train_idx, temp_idx = next(
    splitter1.split(X, y, groups=patients)
)

splitter2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=123
)

val_relative_idx, test_relative_idx = next(
    splitter2.split(
        X.iloc[temp_idx],
        y.iloc[temp_idx],
        groups=patients.iloc[temp_idx]
    )
)

val_idx = temp_idx[val_relative_idx]
test_idx = temp_idx[test_relative_idx]

# Training
X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()

# Validation
X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

# Test
X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()

In [21]:
from pathlib import Path

processed = Path(
    r"C:\Users\faith\OneDrive\Documents\MDX1_MSc_Health_Data_Science\CT4101_Machine_Learning\Machine_Learning_Portfolio\CT4101-Machine-Learning-Portfolio\data\processed"
)

# Save datasets
datasets = {
    "X_train.csv": X_train,
    "X_val.csv": X_val,
    "X_test.csv": X_test,
    "y_train.csv": y_train.to_frame(),
    "y_val.csv": y_val.to_frame(),
    "y_test.csv": y_test.to_frame()
}

for filename, data in datasets.items():
    data.reset_index(drop=True).to_csv(
        processed / filename,
        index=False
    )